# InnerSource Curation Example <a name="top"></a>

## Table of Contents
1. [Introduction](#introduction)
2. [End-to-end Curation Example and Explanations](#example)
    - [Libraries and helper functions](#libraries)
    - [Local paths](#paths)
    - [2.1 Prepare input events](#inputevents)
    - [2.2 Define Curation Event, Processor and Lookup](#extensions)
    - [2.3 Run Curation](#runcuration)
    - [2.4 Review curation outputs](#output)
3. [Conclusion](#conclusion)

## Introduction <a name="introduction"></a>
Curation is the process of standardizing events. During this stage, accepted events are transformed, cleaned, enriched by user-defined processors, and converted into curated events.

<img src="./example/images/curation_example.png" alt="Curation example" width="800"/>

Here is a simple example of an accepted event being curated: An accepted event from the raw file may contain only the full name of an individual in the "fullName" field. After being processed by a user-defined Curation Processor, the name in the curated event is split into "firstName", "middleName", and "lastName", all of which are required for subsequent downstream processing. The curated output now includes events with 6 fields (idType, nationalId, accountNumber, firstName, middleName, and lastName), rather than the 4 found in the accepted event.

For complete details, please refer to the [Oxygen Curation Documentation](https://pages.experian.com/display/ODC/Building+your+pipeline+-+Curation) and [Tutorial](https://pages.experian.com/display/ODC/Oxygen+tutorials%3A++Hands-on+Curation+stage).

[Return to Top](#top)

## End-to-end Curation Example and Explanations <a name="example"></a>
The Curation job orchestrates through the following steps:

1. [**Prepare input events**](#inputevents)
2. [**Define Curation Event, Processor and Lookup**](#extensions)
3. [**Run Curation**](#runcuration)
4. [**Review curation outputs**](#output)

#### Libraries and helper functions <a name="libraries"></a>
First, we begin by importing necessary Python libraries and helper functions. These aid in extracting key information from inputs, intermediary results, and final outputs.

In [1]:
import os
import re
import json
import pandas as pd
pd.set_option('display.max_colwidth', None)

def print_jsonl_1stline(file):
    with open(file, 'r') as f:
        line = f.readline()
        data = json.loads(line)
        print(json.dumps(data, indent=2))

def filter_files(files, substring):
    for file in files:
        if substring in file:
            return file

def print_jsonl_by_eventId(file, eventId):
    with open(file, 'r') as f:
        for line in f:
            data = json.loads(line)
            if data.get("eventId") == eventId:
                print(json.dumps(data, indent=2))
                break

[Return to Top](#top)

#### Local paths <a name="paths"></a>
Subsequently, please revise the path variables to ensure they correctly point to the respective paths of the input, output, and identifying attributes configuration file on your machine.

In [2]:
input_folder = 'example/accepted-input'
output_root_path = 'example/curated-output'

[Return to Top](#top)

### 1. Prepare input events <a name="inputevents"></a>
The inputs for the curation process are accepted events - events that have been validated and accepted. 

Input events usually arrive in batches as folders with different names.

We read from the inputs and display the first event as an example.

In [3]:
# Print the first input event as an example
batch = os.listdir(input_folder)[0]
batch_folder = '/'.join([input_folder, batch])
files = [file for file in os.listdir(batch_folder) if file.endswith('.json')]
file = os.path.join(batch_folder, files[0])
print(f"Batch name: {batch}\n")
print("Sample accepted event:\n")
print_jsonl_1stline(file)

Batch name: E-230001.20161113.T.7042204574991252479

Sample accepted event:

{
  "lineage": {
    "dataSource": "230001",
    "receivedDate": 1485993600000,
    "fileName": "file:/Source/oxygen-is/oxygen-is-data/E-230001.20161113.T",
    "subscriber": "230001",
    "fileAsOfDate": 1485993600000,
    "datasetId": "7042204574991252479",
    "recordHash": "9a7fac1a698e6bc52b33973d9caba16ec8481a63"
  },
  "eventId": -1,
  "validatedRecordJson": "{\"idType\":1,\"nationalId\":\"1001\",\"accountNumber\":\"0000001111111111\",\"fullName\":\"MANUEL JAVIER DUCUARA POTES\",\"situationOfHolder\":0,\"openningDate\":20060503,\"dueDate\":20301231,\"responsibleOrQualityOfDebtor\":0,\"typeOfobligation\":4,\"mortgageSubsidy\":0,\"dateOfSubsidy\":0,\"endOfThatContractCreatesTheObligation\":2,\"wayToPay\":0,\"frequencyOfPayment\":1,\"novelty\":13,\"stateOfOriginAccount\":0,\"stateDateOrigin\":20060503,\"stateOfMind\":5,\"dateStatusAccount\":20161130,\"stateOfPlastics\":0,\"dateStatusPlastic\":0,\"adjective

[Return to Top](#top)

### 2. Define Curation Event, Processor and Lookup <a name="extensions"></a>
The extended CuratedEvent class, such as [CreditCuratedEvent](src/main/scala/com/experian/edf/oxygen/curation/extension/event/CreditCuratedEvent.scala), defines the format of the curated events in the output. 

Similarly, the extended CurationProcessor class, like [CreditCurationProcessorJson](src/main/scala/com/experian/edf/oxygen/curation/extension/processor/CreditCurationProcessorJson.scala), determines how the accepted events are standardized.

In addition, curation fields may utilize reference files as lookup tables (e.g., an official national ID list). Here are some sample lookup files:
- [consistency-table1-creditcard.csv](example/lookup/consistency-table1-creditcard.csv)
- [consistencyfiltertable1.csv](example/lookup/consistencyfiltertable1.csv)
- [data_credito_special_case.csv](example/lookup/data_credito_special_case.csv)

[Return to Top](#top)

### 4. Run Curation <a name="runcuration"></a>
Run Curation with sample input, configurations, extended CurationEvent, extended CurationProcessor, and lookups.

In [4]:
# Uncomment the lines in this block and run pinning
# !cp /data/src/main/resources/log4j.properties /opt/spark/conf/;
# !spark-submit --master local[2] --driver-memory 2g --executor-memory 2g --conf "spark.driver.extraJavaOptions=-Dlog4j.configuration=log4j.properties" --class com.experian.edf.oxygen.curation.app.CurationApp target/oxygen-curation-innersource-1.0-jar-with-dependencies.jar env.conf curation-application.conf oxygen-lookup.conf

[Return to Top](#top)

### 5. Review curation outputs <a name="output"></a>

In [5]:
# Verify output path
output_root_path = 'example/curated-output'
# output_root_path = 'example/output-demo'
output_list = os.listdir(output_root_path)
output_path = os.path.join(output_root_path, 'stock')

# Select an sample event
sample_event_id = '-1'

In [6]:
# Print one of curated event in the output
files = [file for file in os.listdir(output_path) if file.endswith('.json')]
for f in files:
    file = os.path.join(output_path, f)
    print("Sample curated event:\n")
    print_jsonl_by_eventId(file, sample_event_id)

Sample curated event:

{
  "lineage": {
    "dataSource": "230001",
    "receivedDate": 1485993600000,
    "fileName": "file:/Source/oxygen-is/oxygen-is-data/E-230001.20161113.T",
    "subscriber": "230001",
    "fileAsOfDate": 1485993600000,
    "datasetId": "7042204574991252479",
    "recordHash": "9a7fac1a698e6bc52b33973d9caba16ec8481a63"
  },
  "eventId": "-1",
  "eventData": {
    "ints": {},
    "strings": {
      "detailWarranty": "",
      "qualification": "D",
      "middleName": "JAVIER",
      "email": "000000000000230001",
      "directionOfResidenceOrCommercialAddress": "0000265623790000000000000000000000000000Bogota",
      "departmentResidenceOrDepartmentCommercial": "",
      "directionOfCorrespondence": "",
      "nationalId": "1001",
      "departmentLabor": "",
      "firstName": "MANUEL",
      "firstLastName": "DUCUARA",
      "secondLastName": "POTES",
      "departmentOfCorrespondence": "",
      "cityLabor": "gota",
      "accountNumber": "0000001111111111",
   

[Return to Top](#top)

## Conclusion <a name="conclusion"></a>
Through curation, the raw information in the accepted events is transformed into a standard format. Users can define various processors to transform, clean, and enrich the fields, and assign them to different input batches by matching folder name patterns. Users may also utilize reference lookup tables to maintain consistency of reference fields. Ultimately, raw events from diverse sources become well-structured curated events, prepared for further downstream processing.



[Return to Top](#top)